# Labb2 - Pipeline before/after checks

This notebook is for manual verification. Do not add it to the ADF pipeline being checked, and do not use **Run all**.

1. Set `TEST_ID` in the configuration cell. Use a new ID for each experiment, and run the configuration and initialization cells.
2. Before rerunning ADF, run only the **BEFORE** capture cell.
3. Run the entire pipeline in ADF and wait for it to succeed. Avoid concurrent runs from another scheduled pipeline.
4. Copy the **Pipeline Run ID** from ADF Monitor > Pipeline runs into the AFTER cell, then run only that capture cell.
5. Run the **REPORT** cell and capture the results. The report includes the username, ADF Run ID, timestamps, and row counts before and after the run for each table.

Snapshots are stored in `db_labb2.audit.pipeline_check_snapshots` and remain available after compute restarts. You need SELECT access to the source tables, permission to create the audit schema and table, and permission to write to the audit table. If necessary, ask an authorized account to create the audit objects and grant access.

**Important:** Unchanged row counts do not prove that there are no duplicates. Configure verified business keys in `KEY_COLUMNS`. Tables without configured keys show `NOT_CONFIGURED` and do not automatically pass the duplicate check. The configuration leaves business keys unspecified until column names and table grain have been verified.


In [0]:
CATALOG = "db_labb2"
TEST_ID = "labb2_rerun_20261004_01"  # Change for each new experiment; keep the same ID for BEFORE and AFTER.

TABLES = {
    "bronze": [
        "electricity_price_raw", "scb_tab1278", "scb_tab3277",
        "scb_tab5557", "scb_tab5557_backup", "scb_tab638",
    ],
    "silver": ["electricity_prices", "population", "registrations", "vehicles_in_use"],
    "gold": [
        "dim_fuel", "dim_kommun", "dim_price_area", "dim_year",
        "fact_annual_electricity_price", "fact_electricity_price",
        "fact_population", "fact_registrations", "fact_vehicle_stock",
        "mart_charger_priority",
    ],
}

# Configure verified business keys using schema.table names.
# Examples: uncomment only after verifying column names and table grain.
KEY_COLUMNS = {
    # "bronze.electricity_price_raw": ["price_date", "price_area"],
    # "gold.mart_charger_priority": ["kommun_code"],
}

assert TEST_ID.strip(), "TEST_ID cannot be empty"
assert set(KEY_COLUMNS).issubset({f"{s}.{t}" for s, ts in TABLES.items() for t in ts})


In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

def qi(name):
    return "`" + name.replace("`", "``") + "`"

def full_name(schema, table):
    return ".".join(qi(x) for x in (CATALOG, schema, table))

AUDIT_TABLE = full_name("audit", "pipeline_check_snapshots")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qi(CATALOG)}.`audit`")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    test_id STRING,
    phase STRING,
    captured_at TIMESTAMP,
    student_user STRING,
    adf_run_id STRING,
    table_name STRING,
    row_count BIGINT,
    key_columns STRING,
    duplicate_key_count BIGINT,
    null_key_row_count BIGINT
) USING DELTA
""")

def capture_snapshot(phase, adf_run_id=""):
    assert phase in ("before", "after")
    if phase == "after" and not adf_run_id.strip():
        raise ValueError("Fill in the actual ADF Pipeline Run ID before AFTER.")
    existing = spark.table(AUDIT_TABLE).where(F.col("test_id") == TEST_ID)
    if existing.where(F.col("phase") == phase).limit(1).count():
        raise ValueError(f"{TEST_ID}: {phase} already exists. Use a new TEST_ID for a new experiment.")
    if phase == "after" and not existing.where(F.col("phase") == "before").limit(1).count():
        raise ValueError("Capture BEFORE first using the same TEST_ID.")
    student = spark.sql("SELECT current_user() AS u").first()["u"]
    capture_start = datetime.now(timezone.utc)
    rows = []
    # Counts are collected sequentially. Do not run a writer pipeline during capture.
    for schema, tables in TABLES.items():
        for table in tables:
            name = f"{schema}.{table}"
            df = spark.table(full_name(schema, table))
            keys = KEY_COLUMNS.get(name, [])
            duplicate_count = null_count = None
            if keys:
                missing = set(keys) - set(df.columns)
                if missing:
                    raise ValueError(f"{name}: key columns do not exist: {sorted(missing)}")
                duplicate_count = df.groupBy(*[F.col(qi(k)) for k in keys]).count().where(F.col("count") > 1).count()
                null_condition = F.col(qi(keys[0])).isNull()
                for k in keys[1:]:
                    null_condition = null_condition | F.col(qi(k)).isNull()
                null_count = df.where(null_condition).count()
            rows.append((TEST_ID, phase, capture_start, student, adf_run_id.strip(),
                         name, df.count(), ", ".join(keys), duplicate_count, null_count))
    # Write only after all tables have been checked successfully.
    result = spark.createDataFrame(rows, spark.table(AUDIT_TABLE).schema)
    result.write.mode("append").saveAsTable(AUDIT_TABLE)
    display(result.orderBy("table_name"))
    print(f"Saved {phase}: {len(rows)} tables; test_id={TEST_ID}")


## BEFORE - run only before starting ADF

This step reads the source tables and writes audit snapshots. It does not run the business pipeline. Once saved, do not execute this cell again for the same experiment.


In [0]:
capture_snapshot("before")


## AFTER - run only after ADF has succeeded

Enter the actual ADF **Pipeline Run ID**, not the Databricks Run ID. Do not rerun BEFORE to record the results.


In [0]:
ADF_RUN_ID = "be3ed0a7-9d5a-496b-be30-720d64b8bff8"  # Paste the successful ADF Pipeline Run ID here
capture_snapshot("after", ADF_RUN_ID)


## REPORT - capture this result for evidence

`row_delta = 0` means the row count did not change. Newly available source data may produce a positive delta. `duplicate_status = PASS` applies only when the correct business keys are configured and there are no duplicate keys or null key rows after the rerun. `NOT_CONFIGURED` does not prove that a table is free of duplicates.

Each timestamp records the start of the snapshot capture. The 20 tables are read sequentially, not as one atomic snapshot across all tables. Avoid concurrent writes during capture.


In [0]:
snapshots = spark.table(AUDIT_TABLE).where(F.col("test_id") == TEST_ID)
before = snapshots.where(F.col("phase") == "before").select(
    "table_name", F.col("captured_at").alias("before_at"),
    F.col("row_count").alias("rows_before"),
    F.col("key_columns").alias("keys_before"),
    F.col("duplicate_key_count").alias("duplicates_before"),
)
after = snapshots.where(F.col("phase") == "after").select(
    "table_name", "student_user", "adf_run_id",
    F.col("captured_at").alias("after_at"),
    F.col("row_count").alias("rows_after"),
    F.col("key_columns").alias("keys_after"),
    F.col("duplicate_key_count").alias("duplicates_after"),
    F.col("null_key_row_count").alias("null_keys_after"),
)
report = (before.join(after, "table_name", "full")
    .withColumn("test_id", F.lit(TEST_ID))
    .withColumn("row_delta", F.col("rows_after") - F.col("rows_before"))
    .withColumn("duplicate_status",
        F.when(F.col("rows_before").isNull() | F.col("rows_after").isNull(), "INCOMPLETE")
         .when(F.col("keys_after") == "", "NOT_CONFIGURED")
         .when(F.col("keys_before") != F.col("keys_after"), "KEY_CONFIG_CHANGED")
         .when((F.col("duplicates_after") == 0) & (F.col("null_keys_after") == 0), "PASS")
         .otherwise("CHECK"))
    .select("student_user", "test_id", "adf_run_id", "table_name",
            "rows_before", "rows_after", "row_delta", "keys_after",
            "duplicates_before", "duplicates_after", "null_keys_after",
            "duplicate_status", "before_at", "after_at")
    .orderBy("table_name"))
display(report)


## Optional - inspect columns to configure business keys

This cell lists column names; it does not infer unique keys. Verify each key against the table grain and the grouping, merge, or modeling logic in Silver and Gold. Configure `KEY_COLUMNS` before the next experiment.


In [0]:
column_rows = []
for schema, tables in TABLES.items():
    for table in tables:
        column_rows.append((f"{schema}.{table}", ", ".join(spark.table(full_name(schema, table)).columns)))
display(spark.createDataFrame(column_rows, "table_name string, columns string").orderBy("table_name"))
